In [8]:
import os 
%pwd

'd:\\Data Science\\Codes\\02 MLOPs\\End-to-End-ML-Project-with-MLFlow\\research'

In [9]:
os.chdir("../")
%pwd

'd:\\Data Science\\Codes\\02 MLOPs\\End-to-End-ML-Project-with-MLFlow'

In [10]:
from dataclasses import dataclass
from pathlib import Path 

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir : Path 
    train_data_path : Path 
    test_data_path : Path 
    model_name : str 
    C : float
    max_iter : int 
    target_column : str 

In [11]:
from mlProject.constants import * 
from mlProject.utils.common import read_yaml,create_directories

In [12]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self)->ModelTrainerConfig:

        config = self.config.model_trainer
        params = self.params.LogisticRegression
        schema = self.schema.TARGET_COLUMN

        model_trainer_config = ModelTrainerConfig(
            root_dir = config.root_dir ,
            train_data_path = config.train_data_path,
            test_data_path = config.test_data_path,
            model_name = config.model_name ,
            C = params.C,        
            max_iter = params.max_iter,
            target_column = schema.name
        )

        return model_trainer_config

In [13]:
import pandas as pd
import os
from mlProject import logger
from sklearn.linear_model import LogisticRegression 
import joblib

In [14]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    
    def train(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)


        train_x = train_data.drop([self.config.target_column], axis=1)
        test_x = test_data.drop([self.config.target_column], axis=1)
        train_y = train_data[[self.config.target_column]]
        test_y = test_data[[self.config.target_column]]


        lr = LogisticRegression(C=self.config.C,max_iter=self.config.max_iter,random_state=42)
        lr.fit(train_x, train_y)
        # Create directory if it doesn't exist
        os.makedirs(os.path.join(self.config.root_dir), exist_ok=True)
        joblib.dump(lr, os.path.join(self.config.root_dir, self.config.model_name))



In [16]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2026-03-17 06:53:05,127: INFO: common: YAML file loaded successfully: config\config.yaml:]
[2026-03-17 06:53:05,132: INFO: common: YAML file loaded successfully: params.yaml:]
[2026-03-17 06:53:05,142: INFO: common: YAML file loaded successfully: schema.yaml:]
[2026-03-17 06:53:05,146: INFO: common: Directory created at: artifacts:]


d:\Users\Asadullah Core\Apps\AS\envs\mlflow-env\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
